# AMR기반 완성차 스마트 물류 자동화 시스템

## 1. 프로젝트 설명

### 결과 영상: AMR 완성차 물류 운송 시나리오
<img src="./images/AMR.gif" width="1000"/>

### ⚙️ 역할
PM(프로젝트 매니저), AMR 제어 노드 개발, 시스템 아키텍처 설계 및 인터페이스 관리

### ⚙️ 개발 환경
하드웨어 - AMR(TurtleBot4)X2, 충전 스테이션X2, 모니터링 PCX5, 웹 카메라X3

소프트웨어 - Ubuntu 22.04(ROS2 Humble), Python, Flask, SQLite3, OpenCV, YOLOv8, Nav2 패키지, RViz2

### 🔍 프로젝트 개요
완성차 생산 라인에서 물류 창고와 주차장 간에 완성차 자동 이송을 위해 두 대의 AMR(TurtleBot4)을 협력 운용하는 AMR 연동 시스템.

사람의 개입 없이 정밀한 경로 제어 및 위치 인식으로 물류 자동화 구현.

사람이 직접 내려서 주유하는 기존의 무인 주유소 방식과 차별화 되는 완전 자동화 구현.

### 🔍 프로젝트 목표
카메라 인식과 SLAM·ArUco 마커 기반 정밀 도킹으로 적재, 인계, 최종 주차까지 AMR 2대의 협업 과정을 연속적이고 안정적으로 구현.

실제 공장 환경과 유사한 주행 구간 구현 및 실시간 모니터링과 UI를 통해 상태 피드백 제공

### 📊 중요성과 및 문제해결
Aruco 마커 사용 위치 보정을 적용해 차량 인계 지점 AMR 위치 및 자세 오차 감소.

코너 구간 AMR cmd_vel(선속도, 각속도)최적화를 통해 Aruco마커 인식률 향상.

Wi-Fi 고지연으로 Nav2 실행 노드 불안정으로 인한 송수신(pub/sub)문제를 QoS 조정으로 안정화에 실패했지만,

Nav2를 제외한 모듈별 테스트 케이스를 수행해 기한 내 통합 완료.

## 2. 환경 시나리오 구성 및 System 정보

<img src="images/scenario.png" width="1000">


<img src="images/flowchart.png" width="1000">

### 🗂️ AMR 시나리오 표

<img src="images/AMR1_act.png" width="1000">

<img src="images/AMR2_act.png" width="1000">

<img src="images/Topic.png" width="1000">

### 🗂️ System Architecture

<img src="images/SysArch.png" width="1000">

### 🧭 SALM기반 map 생성 과정

<img src="./images/slam.gif" width="1000"/>

## 3. Web GUI(Flask 기반)

### GUI 화면 정보
<img src="images/Flask_GUI.png" width="1000">

###  AMR 공정 사이클 시간 KPI 지표

<img src="images/Flask_KPI.png" width="1000">

## 4. 활용 장치 설명

### AMR - TurtleBot 4(Standard)
<img src="images/turtlebot4.png" width="600">

설명: 기본적으로 9kg의 적재 용량을 제공하고 맞춤 구성 시 최대 15kg까지 확장 가능하며, 최고 속도는 0.306m/s입니다.


### LiDAR 센서: RPLIDAR A1(Low Cost 360 Degree Laser Range Scanner)
<img src="images/lidar.png" width="800">


## 5. AMR2 제어 코드

### ▶️ 설명: 두 노드는 topic 신호(msg)를 주고 받으며 상호적으로 AMR2의 시나리오 별 행동 제어를 실행

### ⚙️ Amr2_system architecture

<img src="images/AMR2_SysArch.png" width="1000">

### A. AMR 이동 제어 

### ▶️ Node: CarRecognitionNavigator(조건 별 AMR2 이동 제어)

### ▶️ 발행 topic: 

robot9/point1 – 인계지점 위치 정밀도 향상을 위한 마커 detect 시작 트리거

robot9/point2 – 하역장 마커 detect 시작 트리거 


### ⚙️ AMR 초기 위치 설정:

현재 충전 스테이지에 dockin상태 확인(X → docking 수행)

초기 pose 설정(set InitialPose)

Nav2 활성화 대기(waitUntilNav2Active)

In [ ]:
import time
import rclpy
from rclpy.node import Node
from std_msgs.msg import Int32, Bool
from turtlebot4_navigation.turtlebot4_navigator import TurtleBot4Directions, TurtleBot4Navigator

class CarRecognitionNavigator(Node):
    def __init__(self): # 위치 설정, 상태변수 초기화, pub/sub 등록
        super().__init__('car_recognition_navigator')
        self.get_logger().info('CarRecognitionNavigator 노드 초기화 시작')

        self.navigator = TurtleBot4Navigator()
        self.goal_pose = []

        # 위치 설정
        self.point1 = self.navigator.getPoseStamped([0.77, -2.8], TurtleBot4Directions.WEST)
        self.point2 = self.navigator.getPoseStamped([0.836, -5.32], TurtleBot4Directions.NORTH)
        self.point3 = self.navigator.getPoseStamped([1.95, -2.74], TurtleBot4Directions.NORTH)

        # 상태 변수 초기화
        self.latest_robot8_value = None
        self.latest_robot9_value = None
        self.is_nothing = False
        self.done1 = False
        self.done2 = False

        # 구독자 등록
        self.create_subscription(Int32, '/robot8/recognized_car', self.robot8_callback, 10)
        self.create_subscription(Int32, '/robot9/recognized_car', self.robot9_callback, 10)
        self.create_subscription(Bool, '/is_nothing', self.is_nothing_callback, 10)
        self.create_subscription(Bool, '/robot9/done1', self.done1_callback, 10)
        self.create_subscription(Bool, '/robot9/done2', self.done2_callback, 10)

        # 발행자 등록
        self.pub_point1 = self.create_publisher(Bool, '/robot9/point1', 10)
        self.pub_point2 = self.create_publisher(Bool, '/robot9/point2', 10)

        self.initialize_navigation()

    def initialize_navigation(self): # 초기 pose 설정 값, Nav2 활성화
        self.get_logger().info('초기 위치 및 Nav2 설정 시작')
        if not self.navigator.getDockedStatus():
            self.get_logger().info('현재 도킹 안됨 → 도킹 수행 중...')
            self.navigator.dock()
        else:
            self.get_logger().info('이미 도킹된 상태')

        initial_pose = self.navigator.getPoseStamped([2.91, -2.59, 0.34], TurtleBot4Directions.NORTH)
        self.navigator.setInitialPose(initial_pose)
        self.navigator.waitUntilNav2Active()
        self.get_logger().info('Nav2 활성화 완료 및 초기 위치 설정됨')

    def is_nothing_callback(self, msg):# msg 없을 경우 상태표시
        self.is_nothing = msg.data
        #self.get_logger().info(f'/is_nothing 수신: {self.is_nothing}')

    def done1_callback(self, msg): # msg 있을 경우 상태표시
        self.done1 = msg.data
        self.get_logger().info('✔️ done1 수신 완료')

    def done2_callback(self, msg): # gotomarker가 발행한 done2 수신 상태표시
        self.done2 = msg.data
        self.get_logger().info('✔️ done2 수신 완료')

    def robot8_callback(self, msg): # AMR1 차량 인계 여부와 종류 확인 후 AMR2 인수인계 지점 이동  
        self.latest_robot8_value = msg.data
        self.get_logger().info(f'/robot8/recognized_car 수신: {msg.data}')

        if msg.data in [1, 2, 3]:
            if self.navigator.getDockedStatus():
                self.get_logger().info('도킹된 상태 → 언도킹 수행')
                self.navigator.undock()

            self.get_logger().info('point1으로 이동 시작')
            self.navigator.startFollowWaypoints([self.point1])
            self.pub_point1.publish(Bool(data=True))
            self.get_logger().info('/robot9/point1 토픽 발행 완료 --2초 기다림')
            time.sleep(2)

    def robot9_callback(self, msg): # gotomarker 실행 여부에 따라 AMR2 이동 제어
        self.latest_robot9_value = msg.data
        self.get_logger().info(f'/robot9/recognized_car 수신: {msg.data}')

        # done1 = True and recognized_car in [1,2,3]
        if self.done1 and msg.data in [1, 2, 3]:
            self.get_logger().info('조건: done1=True && recognized_car=[1,2,3] → point2로 이동 --3초 기다림')
            time.sleep(3)
            self.navigator.startFollowWaypoints([self.point2])
            self.pub_point2.publish(Bool(data=True))
            self.get_logger().info('/robot9/point2 토픽 발행 완료')
            self.done1 = False

        # done2 = True and recognized_car == 0
        elif self.done2 and msg.data == 0:
            self.get_logger().info('조건 확인 중: done2=True && recognized_car=0')
            if self.is_nothing and self.latest_robot8_value == 0:
                self.get_logger().info('조건 만족: is_nothing=True && robot8=0 → point3으로 이동 후 도킹')
                self.navigator.startFollowWaypoints([self.point3])
                self.get_logger().info('2초 기다림')
                time.sleep(2)
                self.navigator.dock()
                self.get_logger().info('도킹 완료')
                self.done2 = False
            else:
                self.get_logger().info('조건 불만족 → point1로 재이동')
                self.navigator.startFollowWaypoints([self.point1])

def main():
    rclpy.init()
    try:
        node = CarRecognitionNavigator()
        rclpy.spin(node)
    except Exception as e:
        print(f'[main] 예외 발생: {e}')
    finally:
        rclpy.shutdown()

if __name__ == '__main__':
    main()

    

### B. ArUco 마커 탐지
### ▶️ Node: ArucoPoseFollower(구역 별 ArUco 마커 인식 및 이동 제어)

### ▶️ 발행 topic:
robot9/done1 – AMR1 ArUco마커 detect 및 이동 완료 신호

robot9/done2 – 차량 하역 완료 신호 

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import Int32, Bool
from sensor_msgs.msg import CompressedImage
from geometry_msgs.msg import Twist
import cv2
import numpy as np

class ArucoPoseFollower(Node):
    def __init__(self): # sub/pub 설정, ArUco마커 세팅, AMR PD제어 초기화
        super().__init__('aruco_pose_follower')

        # Subscribers
        self.image_sub = self.create_subscription(
            CompressedImage,
            '/robot9/oakd/rgb/image_raw/compressed',
            self.image_callback,
            10
        )

        self.car_id_sub = self.create_subscription(Int32, '/robot9/recognized_car', self.car_id_callback, 10)
        self.point1_sub = self.create_subscription(Bool, '/robot9/point1', self.point1_callback, 10)
        self.point2_sub = self.create_subscription(Bool, '/robot9/point2', self.point2_callback, 10)

        # Publishers
        self.cmd_pub = self.create_publisher(Twist, '/robot9/cmd_vel', 10)
        self.done1_pub = self.create_publisher(Bool, '/robot9/done1', 10)
        self.done2_pub = self.create_publisher(Bool, '/robot9/done2', 10)

        # ArUco settings
        self.aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        self.aruco_params = cv2.aruco.DetectorParameters()
        self.camera_matrix = np.array([[620.0, 0.0, 320.0], [0.0, 620.0, 240.0], [0.0, 0.0, 1.0]])
        self.dist_coeffs = np.zeros((5, 1))
        self.marker_length = 0.05  # meters
        self.target_distance = 0.30  # meters

        # PD control
        self.Kp = 0.002
        self.Kd = 0.0008
        self.previous_error = 0.0
        self.previous_time = self.get_clock().now()

        # State
        self.mode = None  # 'point1', 'point2'
        self.car_id = 0
        self.target_marker_id = None

        self.get_logger().info("✅ ArUco Pose Follower Node Initialized")

    def point1_callback(self, msg): # point1 명령 수신 시, 마커 AMR1 부각 마커 추적 시작
        if msg.data:
            self.mode = 'point1'
            self.target_marker_id = 8
            self.get_logger().info("🚩 point1 명령 수신: 마커 ID 8 추적 시작")

    def point2_callback(self, msg): # point2 명령 수신 시, 하역장 이동 시작 
        if msg.data:
            self.mode = 'point2'
            self.target_marker_id = self.car_id
            self.get_logger().info(f"🚩 point2 명령 수신: 마커 ID {self.car_id} 추적 시작")

    def car_id_callback(self, msg): # 인식된 차량 ID 저장
        self.car_id = msg.data
        self.get_logger().info(f"🎯 인식된 차량 ID: {self.car_id}")

    def image_callback(self, msg): # 이미지 수신 후 하역장 ArUco 마커 추적 및 이동
        if self.mode is None or self.target_marker_id is None:
            return

        np_arr = np.frombuffer(msg.data, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
        corners, ids, _ = cv2.aruco.detectMarkers(frame, self.aruco_dict, parameters=self.aruco_params)

        twist = Twist()

        if ids is not None and self.target_marker_id in ids:
            index = np.where(ids == self.target_marker_id)[0][0]
            marker_corners = [corners[index]]

            rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
                marker_corners,
                self.marker_length,
                self.camera_matrix,
                self.dist_coeffs
            )

            tvec = tvecs[0][0]
            distance = tvec[2]
            c = corners[index][0]
            center_x = int(np.mean(c[:, 0]))
            error = center_x - (frame.shape[1] // 2)

            current_time = self.get_clock().now()
            dt = (current_time - self.previous_time).nanoseconds / 1e9
            derivative = (error - self.previous_error) / dt if dt > 0 else 0.0
            angular_z = -(self.Kp * error + self.Kd * derivative)

            twist.angular.z = angular_z
            self.previous_error = error
            self.previous_time = current_time

            if distance > self.target_distance:
                twist.linear.x = min(0.15, 0.5 * (distance - self.target_distance))
            else:
                twist.linear.x = 0.0
                twist.angular.z = 0.0
                self.cmd_pub.publish(twist)

                done_msg = Bool()
                done_msg.data = True
                if self.mode == 'point1':
                    self.done1_pub.publish(done_msg)
                    self.get_logger().info("✅ point1 목표 도달. /robot9/done1 = True 발행")
                elif self.mode == 'point2':
                    self.done2_pub.publish(done_msg)
                    self.get_logger().info("✅ point2 목표 도달. /robot9/done2 = True 발행")

                # Reset
                self.mode = None
                self.target_marker_id = None
                return

            self.get_logger().info(f'[ID {self.target_marker_id}] 거리: {distance:.2f} m | 오차: {error} px | 회전속도: {angular_z:.3f}')
            cv2.aruco.drawDetectedMarkers(frame, marker_corners)

        else:
            twist.linear.x = 0.0
            twist.angular.z = 0.005
            self.get_logger().info(f'🔍 마커 ID {self.target_marker_id} 탐색 중...')

        self.cmd_pub.publish(twist)

        cv2.imshow('Aruco Pose Tracking', frame)
        cv2.waitKey(1)

    def destroy_node(self): # 노드 종료 시 OpenCV 윈도우 닫기
        cv2.destroyAllWindows()
        super().destroy_node()

def main():
    rclpy.init()
    node = ArucoPoseFollower()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()


